# 04 - Analyse de la Masse des Neutrinos: LambdaCDM vs Tachyon

Ce notebook a pour objectif de vérifier si le champ Tachyon permet de résoudre l'anomalie cosmologique récente sur la masse des neutrinos. Sous le modèle standard LambdaCDM, les données récentes ont tendance à "écraser" la masse des neutrinos vers des valeurs impossibles en physique des particules (proches de 0 eV, voire négatives).

Nous allons vérifier si l'introduction du modèle géométrique de Souriau (le champ Tachyon) absorbe cette tension et permet aux neutrinos de retrouver une masse physiquement valide (supérieure à la limite expérimentale de ~0.06 eV).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, loadMCSamples

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,'axes.grid': True, 'grid.alpha': 0.3})

print("Environnement préparé pour l'analyse du volet Neutrino.")

Entorno preparado para análisis de Neutrinos.


## 1. Préparation de l'environnement de travail

Dans cette première étape, nous préparons notre environnement Python. Nous utilisons GetDist, la librairie de référence en cosmologie développée pour analyser et tracer les résultats des chaînes de Markov (MCMC). Elle gère nativement la pondération (les weights) et le lissage des distributions statistiques.

In [ ]:
ruta_lcdm_meff = '/content/drive/MyDrive/souriau_chains/neutrino_meff/lcdm_meff'
ruta_tachyon_meff = '/content/drive/MyDrive/souriau_chains/neutrino_meff/tachyon_meff'

try:
    print("Chargement des échantillons ΛCDM (Masse effective)...")
    muestras_lcdm = loadMCSamples(ruta_lcdm_meff, settings={'ignore_rows': 0.3})

    print("Chargement des échantillons Tachyon (Masse effective)...")
    muestras_tach = loadMCSamples(ruta_tachyon_meff, settings={'ignore_rows': 0.3})
except Exception as e:
    print(f"Erreur de chargement (Normal si les runs n'ont pas encore été lancés) : {e}")
    raise SystemExit

# le paramètre tel que défini dans le futur neutrino_meff.yaml
PARAM_NEUTRINO = 'mnu_eff'  

Cargando muestras LambdaCDM...
Cargando muestras Tachyon...


## 2. Chargement des Chaînes MCMC et Application du Burn-in

Nous importons ici les données brutes générées par nos exécutions sur Cobaya (le run de référence LambdaCDM et notre run Tachyon).

Pourquoi appliquer un "burn-in" de 30%?
Lorsqu'un algorithme MCMC démarre, il passe ses premiers milliers de pas à "chercher" la vallée où les paramètres sont optimaux. Cette phase d'exploration initiale ne représente pas la vraie probabilité finale. En Data Science bayésienne, il est impératif de couper (ignorer) ce début de chaîne pour ne calculer nos statistiques que sur la marche aléatoire stable (es un mero primer approach al espacio de búsqueda).

In [ ]:
stats_lcdm = muestras_lcdm.getMargeStats()
stats_tach = muestras_tach.getMargeStats()

media_lcdm = stats_lcdm.parWithName(PARAM_NEUTRINO).mean
err_lcdm = stats_lcdm.parWithName(PARAM_NEUTRINO).err

media_tach = stats_tach.parWithName(PARAM_NEUTRINO).mean
err_tach = stats_tach.parWithName(PARAM_NEUTRINO).err

print("RÉSULTATS DU SHIFT DES NEUTRINOS (Anomalie DESI)")
print(f"Modèle ΛCDM: Σm_ν^eff = {media_lcdm:+.4f} ± {err_lcdm:.4f} eV")
print(f"Modèle Tachyon: Σm_ν^eff = {media_tach:+.4f} ± {err_tach:.4f} eV")

shift = media_tach - media_lcdm
print(f"SHIFT DÉTECTÉ: {shift:+.4f} eV")

# logique de décision de la "Couche Basse" (Souriau)
if media_lcdm < 0.0 and media_tach > 0.05:
    print("\nANOMALIE RÉSOLUE !")
    print("Sous ΛCDM, les données forcent une masse effective négative (non-physique).")
    print("L'ajout du Tachyon absorbe la tension et restaure une masse positive saine.")
    print("C'est la signature recherchée par le modèle géométrique.")


Marginalized limits: 0.68; 0.95; 0.99

parameter                                  mean           sddev          lower1         upper1         limit1 lower2         upper2         limit2 lower3         upper3         limit3 
H0                                         6.8358009E+01  3.0660548E-01  6.8085872E+01  6.8681975E+01  two    6.7719845E+01  6.8934428E+01  two    6.7590686E+01  6.9211112E+01  two     H0
ombh2                                      2.2521006E-02  1.2396061E-04  2.2402865E-02  2.2646364E-02  two    2.2267497E-02  2.2765251E-02  two    2.2174317E-02  2.2842444E-02  two     ombh2
omch2                                      1.1772077E-01  6.8910943E-04  1.1701736E-01  1.1833386E-01  two    1.1636545E-01  1.1919041E-01  two    1.1572735E-01  1.1989283E-01  two     omch2
As                                         2.1012106E-09  3.4818816E-11  2.0659856E-09  2.1372558E-09  two    2.0390370E-09  2.1647896E-09  two    2.0106630E-09  2.1892110E-09  two     As
ns                

AttributeError: 'NoneType' object has no attribute 'mean'

## 3. Preuve Empirique de la Théorie

Dans la prochaine cellule nous testons numériquement l'hypothèse de la "couche basse" de la théorie de Souriau.
Nous extrayons la médiane et l'erreur (les statistiques marginalisées) pour la somme des masses des neutrinos dans les deux univers simulés.

Notre algorithme inclut une règle de décision stricte: si la masse médiane sous LambdaCDM est poussée sous les 0.03 eV (irréaliste), mais que le passage au modèle Tachyon permet à la masse de "respirer" et de remonter au-dessus de 0.05 / 0.06 eV (limites empiriques), nous obtenons la preuve statistique que le champ Tachyon absorbe la pathologie du modèle standard.

In [ ]:
g = plots.get_subplot_plotter()

params_comparaison = ['H0', 'Omega_m', PARAM_NEUTRINO]

g.triangle_plot(
    [muestras_lcdm, muestras_tach],
    params_comparaison,
    filled=True,
    legend_labels=['$\\Lambda$CDM (Masse Libre)', 'Tachyon (Masse Libre)'],
    contour_colors=['#1f77b4', '#d62728'],
    title_limit=1 
)

# ligne du zéro physique (pour bien montrer l'anomalie LCDM)
for i in range(len(params_comparaison)):
    ax = g.subplots[i, i]
    if ax is not None and params_comparaison[i] == PARAM_NEUTRINO:
        ax.axvline(0.0, color='black', linestyle='-', lw=2, label='Zéro Physique')
        ax.axvline(0.06, color='gray', linestyle='--', lw=2, label='Limite standard (0.06 eV)')
        ax.legend(loc='upper right', fontsize=9)

plt.suptitle("Résolution de l'anomalie de masse des neutrinos via le champ Tachyon", fontsize=16, y=1.05)
plt.show()

## 4. Visualisation Globale : Le "Corner Plot" Comparatif

Ce type de graphique superpose les distributions de probabilité a posteriori (les "cloches") en 1D, et les ellipses de corrélation en 2D.

Nous traçons une ligne de démarcation physique stricte à 0.06 eV, qui correspond à la limite basse mesurée par les expériences d'oscillation de particules sur Terre. L'objectif visuel est d'observer la cloche bleue (LambdaCDM) s'écraser à gauche de cette ligne, tandis que la cloche rouge (Tachyon) bascule naturellement à droite, dans la zone physiquement autorisée.

In [ ]:
param_taquion = 'tachyon_lambda'

if param_taquion in muestras_tach.getParamNames().list():
    g2 = plots.get_single_plotter(width_inch=6)
    g2.plot_2d(muestras_tach, param_taquion, PARAM_NEUTRINO, filled=True, colors=['#d62728'])
    
    g2.add_y_marker(0.06, color='gray', ls='--', lw=2)
    
    plt.title("Dégénérescence : Tachyon $\\lambda$ vs $\\Sigma m_\\nu$")
    plt.savefig('/content/drive/MyDrive/souriau_chains/degeneracy_tachyon_nu.png', bbox_inches='tight', dpi=300)
    plt.show()